# 05 - Comparison with scikit-learn

Now running the same problem through sklearn's LogisticRegression, and comparing it properly against my from-scratch version - accuracy, precision, recall, F1, confusion matrix, ROC-AUC.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_auc_score, roc_curve)

np.random.seed(42)

## Rebuilding everything (same as notebook 04)

In [ ]:
df = pd.read_csv('../data/raw/student-mat.csv', sep=';')
df['pass'] = (df['G3'] >= 10).astype(int)

df_model = df.drop(columns=['G1', 'G2', 'G3'])

binary_cols = ['schoolsup', 'famsup', 'paid', 'activities', 'nursery',
               'higher', 'internet', 'romantic']
for col in binary_cols:
    df_model[col] = df_model[col].map({'yes': 1, 'no': 0})

df_model['school'] = df_model['school'].map({'GP': 1, 'MS': 0})
df_model['sex'] = df_model['sex'].map({'F': 1, 'M': 0})
df_model['address'] = df_model['address'].map({'U': 1, 'R': 0})
df_model['famsize'] = df_model['famsize'].map({'GT3': 1, 'LE3': 0})
df_model['Pstatus'] = df_model['Pstatus'].map({'T': 1, 'A': 0})

multi_cat_cols = ['Mjob', 'Fjob', 'reason', 'guardian']
df_model = pd.get_dummies(df_model, columns=multi_cat_cols, drop_first=True)
df_model = df_model.astype(float)

X = df_model.drop(columns=['pass']).values
y = df_model['pass'].values

def train_test_split_manual(X, y, test_size=0.2, seed=42):
    np.random.seed(seed)
    n = X.shape[0]
    indices = np.random.permutation(n)
    test_count = int(n * test_size)
    test_idx = indices[:test_count]
    train_idx = indices[test_count:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split_manual(X, y)

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1
X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

## My from-scratch model (copied from notebook 04)

Copying the functions here too so this notebook can run on its own without depending on the previous one.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gradient_descent(X, y, learning_rate=0.1, n_iterations=2000):
    n_samples, n_features = X.shape
    weights = np.zeros(n_features)
    bias = 0

    for i in range(n_iterations):
        z = np.dot(X, weights) + bias
        predictions = sigmoid(z)
        dw = (1/n_samples) * np.dot(X.T, (predictions - y))
        db = (1/n_samples) * np.sum(predictions - y)
        weights -= learning_rate * dw
        bias -= learning_rate * db

    return weights, bias

def predict_scratch(X, weights, bias, threshold=0.5):
    probs = sigmoid(np.dot(X, weights) + bias)
    return (probs >= threshold).astype(int), probs

weights, bias = gradient_descent(X_train_scaled, y_train)
y_pred_scratch, y_probs_scratch = predict_scratch(X_test_scaled, weights, bias)

## scikit-learn's version

In [ ]:
sk_model = LogisticRegression(max_iter=2000)
sk_model.fit(X_train_scaled, y_train)

y_pred_sklearn = sk_model.predict(X_test_scaled)
y_probs_sklearn = sk_model.predict_proba(X_test_scaled)[:, 1]

## Side-by-side comparison

In [ ]:
def print_metrics(name, y_true, y_pred, y_probs):
    print(f'--- {name} ---')
    print(f'Accuracy:  {accuracy_score(y_true, y_pred):.3f}')
    print(f'Precision: {precision_score(y_true, y_pred):.3f}')
    print(f'Recall:    {recall_score(y_true, y_pred):.3f}')
    print(f'F1 score:  {f1_score(y_true, y_pred):.3f}')
    print(f'ROC-AUC:   {roc_auc_score(y_true, y_probs):.3f}')
    print()

print_metrics('From scratch (NumPy)', y_test, y_pred_scratch, y_probs_scratch)
print_metrics('scikit-learn', y_test, y_pred_sklearn, y_probs_sklearn)

Expecting these two to be pretty close - if my implementation is correct they should land in a similar range, sklearn might be a bit better since it uses a more advanced optimizer (not plain gradient descent) and has regularization by default.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

cm_scratch = confusion_matrix(y_test, y_pred_scratch)
cm_sklearn = confusion_matrix(y_test, y_pred_sklearn)

sns.heatmap(cm_scratch, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('From scratch - Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_sklearn, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('scikit-learn - Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

(forgot to import seaborn up top for the heatmap, adding it in the cell above's imports section would be cleaner but leaving a note here - need to add `import seaborn as sns` to the first cell)

In [ ]:
fpr_scratch, tpr_scratch, _ = roc_curve(y_test, y_probs_scratch)
fpr_sklearn, tpr_sklearn, _ = roc_curve(y_test, y_probs_sklearn)

plt.figure(figsize=(6,5))
plt.plot(fpr_scratch, tpr_scratch, label='From scratch')
plt.plot(fpr_sklearn, tpr_sklearn, label='scikit-learn')
plt.plot([0,1], [0,1], linestyle='--', color='gray', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

## Comparing the actual coefficients

Since both models are logistic regression, they should learn similar coefficients for each feature (after accounting for sklearn's regularization). Let's check which features both models think matter most.

In [ ]:
feature_names = df_model.drop(columns=['pass']).columns

comparison = pd.DataFrame({
    'feature': feature_names,
    'scratch_coef': weights,
    'sklearn_coef': sk_model.coef_[0]
})

comparison['abs_scratch'] = comparison['scratch_coef'].abs()
comparison.sort_values('abs_scratch', ascending=False).head(10)

## Next steps

This was the last modeling notebook. Left to do:
- write the real README with actual findings (not the placeholder one)
- write the final PDF report